# NCAA March Madness — Exploratory Data Analysis

This notebook explores the Kaggle March ML Mania dataset (2003–2026) and validates our feature pipeline.

**Sections:**
1. Data Overview & Quality Check
2. Tournament Structure & Seed Analysis
3. Rating Systems Comparison
4. Momentum Features
5. Feature Correlation & Matrix Summary

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_collection import load_csv, load_all_mens_data
from src.pipeline import build_feature_matrix, _parse_seed_num

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

%matplotlib inline

## 1. Data Overview & Quality Check

In [ ]:
data = load_all_mens_data()
print("Loaded datasets:")
for key, df in data.items():
    print(f"  {key:20s} {str(df.shape):>15s}  cols: {list(df.columns[:5])}...")

In [ ]:
# Season coverage
tourney = data['tourney_compact']
regular = data['regular_compact']

season_stats = pd.DataFrame({
    'regular_games': regular.groupby('Season').size(),
    'tourney_games': tourney.groupby('Season').size(),
})

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
season_stats['regular_games'].plot(kind='bar', ax=axes[0], color='steelblue', alpha=0.8)
axes[0].set_title('Regular Season Games per Year')
axes[0].set_ylabel('Number of Games')

season_stats['tourney_games'].plot(kind='bar', ax=axes[1], color='coral', alpha=0.8)
axes[1].set_title('Tournament Games per Year')
axes[1].set_ylabel('Number of Games')

plt.tight_layout()
plt.savefig('../output/games_per_season.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nTotal regular season games: {len(regular):,}")
print(f"Total tournament games: {len(tourney):,}")
print(f"Seasons covered: {tourney['Season'].min()} - {tourney['Season'].max()}")

## 2. Tournament Structure & Seed Analysis

In [ ]:
# Seed vs win rate analysis
seeds = data['seeds'].copy()
seeds['SeedNum'] = seeds['Seed'].apply(_parse_seed_num)

# Merge seeds onto tournament results
t = tourney.merge(seeds[['Season', 'TeamID', 'SeedNum']], 
                  left_on=['Season', 'WTeamID'], right_on=['Season', 'TeamID'])\
           .rename(columns={'SeedNum': 'WSeed'}).drop(columns='TeamID')
t = t.merge(seeds[['Season', 'TeamID', 'SeedNum']], 
            left_on=['Season', 'LTeamID'], right_on=['Season', 'TeamID'])\
     .rename(columns={'SeedNum': 'LSeed'}).drop(columns='TeamID')

# Upset = lower seed (higher number) wins
t['Upset'] = t['WSeed'] > t['LSeed']
t['SeedDiff'] = abs(t['WSeed'] - t['LSeed'])

print(f"Overall upset rate: {t['Upset'].mean():.1%}")
print(f"\nUpset rate by seed difference:")
upset_by_diff = t.groupby('SeedDiff')['Upset'].agg(['mean', 'count'])
upset_by_diff.columns = ['upset_rate', 'n_games']
print(upset_by_diff.head(16).to_string())

In [ ]:
# Win rate by seed number
wins_by_seed = t.groupby('WSeed').size().reset_index(name='wins')
losses_by_seed = t.groupby('LSeed').size().reset_index(name='losses')

seed_perf = wins_by_seed.merge(losses_by_seed, left_on='WSeed', right_on='LSeed', how='outer')
seed_perf['Seed'] = seed_perf['WSeed'].fillna(seed_perf['LSeed']).astype(int)
seed_perf = seed_perf.fillna(0)
seed_perf['win_rate'] = seed_perf['wins'] / (seed_perf['wins'] + seed_perf['losses'])
seed_perf = seed_perf.sort_values('Seed')

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(seed_perf['Seed'], seed_perf['win_rate'], color='steelblue', alpha=0.8)
ax.axhline(y=0.5, color='red', linestyle='--', alpha=0.5, label='50%')
ax.set_xlabel('Seed Number')
ax.set_ylabel('Win Rate')
ax.set_title('Tournament Win Rate by Seed (2003-2026)')
ax.set_xticks(range(1, 17))
ax.legend()
plt.savefig('../output/seed_win_rate.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Classic matchup heatmap: seed A vs seed B win rate
matchup_wins = pd.crosstab(t['WSeed'], t['LSeed'])
matchup_total = matchup_wins.copy()
# Symmetrize
for i in range(1, 17):
    for j in range(1, 17):
        if i in matchup_wins.index and j in matchup_wins.columns:
            w = matchup_wins.loc[i, j] if (i in matchup_wins.index and j in matchup_wins.columns) else 0
        else:
            w = 0
        if j in matchup_wins.index and i in matchup_wins.columns:
            l = matchup_wins.loc[j, i] if (j in matchup_wins.index and i in matchup_wins.columns) else 0
        else:
            l = 0
        total = w + l
        if i <= 16 and j <= 16:
            matchup_total.loc[i, j] = w / total if total > 0 else 0.5

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(matchup_total.loc[1:16, 1:16], annot=True, fmt='.2f', 
            cmap='RdYlGn', center=0.5, ax=ax, vmin=0, vmax=1)
ax.set_xlabel('Losing Seed')
ax.set_ylabel('Winning Seed')
ax.set_title('Win Rate Heatmap: Row Seed vs Column Seed')
plt.savefig('../output/seed_matchup_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Rating Systems Comparison

In [ ]:
# How many ranking systems are available per season?
massey = data['massey']
systems_per_season = massey.groupby('Season')['SystemName'].nunique()

fig, ax = plt.subplots(figsize=(12, 5))
systems_per_season.plot(kind='bar', ax=ax, color='steelblue', alpha=0.8)
ax.set_title('Number of Ranking Systems per Season')
ax.set_ylabel('Number of Systems')
plt.tight_layout()
plt.show()

# Key systems availability
key_systems = ['POM', 'SAG', 'MOR', 'DOL', 'COL', 'RPI', 'AP', 'USA', 'WOL']
for sys in key_systems:
    seasons_with = massey[massey['SystemName'] == sys]['Season'].nunique()
    print(f"  {sys:5s}: available in {seasons_with} seasons")

In [ ]:
# Correlation between rating systems (using 2025 as example)
from src.pipeline import build_rating_features_for_season

ratings_2025 = build_rating_features_for_season(data, 2025)
print(f"Rating systems available for 2025: {list(ratings_2025.columns)}")
print(f"Teams with ratings: {len(ratings_2025)}")

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(ratings_2025.corr(), annot=True, fmt='.2f', cmap='coolwarm', ax=ax)
ax.set_title('Correlation Between Rating Systems (2025)')
plt.savefig('../output/rating_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Momentum Features

In [ ]:
# Distribution of momentum features in the full feature matrix
X, y = build_feature_matrix(data, list(range(2014, 2027)))

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Win pct diff
for label, mask, color in [('TeamA wins', y == 1, 'green'), ('TeamA loses', y == 0, 'red')]:
    axes[0].hist(X.loc[mask, 'momentum_winpct_diff'], bins=30, alpha=0.5, label=label, color=color)
axes[0].set_xlabel('Momentum Win% Diff (A - B)')
axes[0].set_title('Momentum Win% Diff by Outcome')
axes[0].legend()

# Margin diff
for label, mask, color in [('TeamA wins', y == 1, 'green'), ('TeamA loses', y == 0, 'red')]:
    axes[1].hist(X.loc[mask, 'momentum_margin_diff'], bins=30, alpha=0.5, label=label, color=color)
axes[1].set_xlabel('Momentum Margin Diff (A - B)')
axes[1].set_title('Momentum Margin Diff by Outcome')
axes[1].legend()

# Seed diff
for label, mask, color in [('TeamA wins', y == 1, 'green'), ('TeamA loses', y == 0, 'red')]:
    axes[2].hist(X.loc[mask, 'seed_diff'], bins=30, alpha=0.5, label=label, color=color)
axes[2].set_xlabel('Seed Diff (A - B)')
axes[2].set_title('Seed Diff by Outcome')
axes[2].legend()

plt.tight_layout()
plt.savefig('../output/feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Feature Correlation & Matrix Summary

In [ ]:
print(f"Feature matrix shape: {X.shape}")
print(f"Target distribution: {pd.Series(y).value_counts().to_dict()}")
print(f"\nMissing values per feature:")
missing = X.isnull().sum()
print(missing[missing > 0].sort_values(ascending=False).to_string())

print(f"\nFeature statistics:")
X.describe().T[['mean', 'std', 'min', 'max']]

In [ ]:
# Correlation of diff features with outcome
diff_cols = [c for c in X.columns if 'diff' in c]
corr_with_target = X[diff_cols].corrwith(pd.Series(y, index=X.index)).sort_values()

fig, ax = plt.subplots(figsize=(10, 6))
corr_with_target.plot(kind='barh', ax=ax, color=['coral' if v < 0 else 'steelblue' for v in corr_with_target])
ax.set_xlabel('Correlation with TeamA Win')
ax.set_title('Feature-Outcome Correlation (Diff Features)')
ax.axvline(x=0, color='black', linewidth=0.5)
plt.tight_layout()
plt.savefig('../output/feature_target_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Full feature correlation heatmap (diff features only for readability)
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(X[diff_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm', ax=ax, center=0)
ax.set_title('Feature Correlation Matrix (Diff Features)')
plt.tight_layout()
plt.savefig('../output/feature_correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Save feature matrix for downstream use
X['target'] = y
X.to_csv('../output/feature_matrix.csv', index=False)
print(f"Feature matrix saved to output/feature_matrix.csv ({X.shape[0]} rows, {X.shape[1]} cols)")